# 실습 2 · RAG와 GraphRAG로 논문에 질문하기

같은 논문 다섯 편에 **기존 RAG**(벡터 검색)와 **GraphRAG**(지식 그래프 + 커뮤니티 요약)를 만들고, 같은 질문으로 비교한다.

| 단계 | 내용 |
|---|---|
| 2 | 논문 불러오기 (Europe PMC의 CC BY 오픈액세스 논문, 실행할 때 내려받음) |
| 3–5 | 기존 RAG: 청킹 → 임베딩 → 코사인 유사도 → 근거가 붙은 답 |
| 6 | 온톨로지(노드 종류 + 허용된 관계)에 맞춰 개체·관계 추출, 실습 1 레코드 추가 |
| 7–9 | GraphRAG: Leiden 커뮤니티 → 커뮤니티 요약 → map-reduce 답 |
| 10–11 | 내 질문 세 개로 두 방법을 판정 |

> API 호출: 한 번 끝까지 실행하면 약 60–80회, 분당 10회로 천천히 호출하므로 10분 안팎 걸린다.
> 같은 입력은 저장해 두었다가 다시 쓰므로, 다시 실행하면 거의 바로 끝난다.
> Europe PMC에 접속할 수 없으면 실습 1의 미니 논문으로 자동 전환한다.

In [ ]:
#@title 1. 환경 설정 — 한 번만 실행 (펼치면 도우미 코드)

# 이 셀은 실습에 필요한 도우미 함수를 정의한다. 고칠 필요는 없다.

# ── 공통 도우미: 환경 확인 · API 키 · Gemini 호출 · 캐시 · 속도 제한 ──────────────
# Colab, 로컬 Jupyter, VS Code 어디서나 같은 코드로 돈다.
import hashlib
import importlib
import json
import os
import pathlib
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
MODEL = "gemini-3.5-flash-lite"            # 막히면 FALLBACK_MODEL로 자동 전환
FALLBACK_MODEL = "gemini-3.1-flash-lite"
EMBED_MODEL = "gemini-embedding-001"       # 막히면 "tfidf"로 바꿔 API 없이 진행
RPM = 10                                   # 무료 티어 분당 호출 한도에 맞춘 간격
WORK = pathlib.Path("/content/llm_practice" if IN_COLAB else "llm_practice_work")
_client = None
_last_call = [0.0]


def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


def ensure_packages(extra=()):
    """필요한 패키지가 없으면 설치한다 (Colab·로컬 공통)."""
    import importlib.metadata as md
    import importlib.util
    need = {"google.genai": "google-genai>=2.20,<3", "pydantic": "pydantic>=2", "pandas": "pandas",
            "matplotlib": "matplotlib", "numpy": "numpy"}
    need.update(dict(extra))
    missing = []
    for mod, pkg in need.items():
        try:
            found = importlib.util.find_spec(mod) is not None
        except ModuleNotFoundError:
            found = False
        if not found:
            missing.append(pkg)
    try:                                   # Colab에 깔린 google-genai가 오래됐으면 올린다
        if int(md.version("google-genai").split(".")[0]) < 2 and "google-genai>=2.20,<3" not in missing:
            missing.append("google-genai>=2.20,<3")
    except md.PackageNotFoundError:
        pass
    if missing:
        print("설치 중:", ", ".join(missing))
        _pip(missing)
        importlib.invalidate_caches()
    # 기관·회사 네트워크의 인증서 검사 대응 (있으면 쓰고, 없으면 넘어간다)
    try:
        import truststore
        truststore.inject_into_ssl()
    except Exception:
        pass


def _read_dotenv():
    for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        f = d / ".env"
        if f.is_file():
            for line in f.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith(("GEMINI_API_KEY=", "GOOGLE_API_KEY=")):
                    return line.split("=", 1)[1].strip().strip("'\"")
    return None


def load_api_key():
    """환경변수 → .env 파일 → Colab 보안 비밀 → 직접 입력 순서로 찾는다."""
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    src = "환경변수"
    if not key:
        key, src = _read_dotenv(), ".env 파일"
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key, src = userdata.get("GEMINI_API_KEY"), "Colab 보안 비밀"
        except Exception:
            key = None
    if not key:
        import getpass
        key, src = getpass.getpass("Gemini API 키를 붙여 넣으세요 (화면에 보이지 않음): ").strip(), "직접 입력"
    if not key:
        raise RuntimeError("API 키를 찾지 못했다. README의 'API 키 준비'를 따라 키를 등록한다.")
    os.environ["GEMINI_API_KEY"] = key
    return key, src


def gemini_client():
    """키를 찾아 Gemini 클라이언트를 만든다. '키 확인: 찾음'이 나오면 된다."""
    global _client
    from google import genai
    from google.genai import types
    key, src = load_api_key()
    _client = genai.Client(api_key=key, http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(attempts=4, initial_delay=8.0)))
    print(f"키 확인: 찾음 ({src}, …{key[-4:]}) · 모델 {MODEL}")
    return _client


def _cache_path():
    WORK.mkdir(parents=True, exist_ok=True)
    return WORK / "llm_cache.jsonl"


def _cache_load():
    p, out = _cache_path(), {}
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            try:
                d = json.loads(line)
                out[d["k"]] = d["v"]
            except Exception:
                pass
    return out


_CACHE = None


def _cached(key):
    global _CACHE
    if _CACHE is None:
        _CACHE = _cache_load()
    return _CACHE.get(key)


def _store(key, value):
    _CACHE[key] = value
    with open(_cache_path(), "a", encoding="utf-8") as f:
        f.write(json.dumps({"k": key, "v": value}, ensure_ascii=False) + "\n")


def _throttle():
    gap = 60.0 / max(RPM, 1)
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()


def llm(prompt, schema=None, image=None, model=None):
    """Gemini 호출 한 번. schema(pydantic 모델)를 주면 그 형식의 객체를, 아니면 글자를 돌려준다.
    같은 입력은 캐시에서 꺼내므로 다시 실행해도 호출 수가 늘지 않는다."""
    from google.genai import types
    if _client is None:
        gemini_client()
    model = model or MODEL
    h = hashlib.sha256()
    h.update((model + "\n" + prompt).encode("utf-8"))
    if schema is not None:
        h.update(json.dumps(schema.model_json_schema(), sort_keys=True).encode())
    if image is not None:
        h.update(image)
    key = h.hexdigest()
    hit = _cached(key)
    if hit is None:
        contents = [types.Part.from_bytes(data=image, mime_type="image/png"), prompt] if image is not None else prompt
        cfg = None
        if schema is not None:
            cfg = types.GenerateContentConfig(response_mime_type="application/json",
                                              response_json_schema=schema.model_json_schema())
        last_err = None
        for m in (model, FALLBACK_MODEL):
            try:
                _throttle()
                resp = _client.models.generate_content(model=m, contents=contents, config=cfg)
                hit = resp.text
                break
            except Exception as e:                       # 쿼터 소진 등 → 대체 모델
                last_err = e
                print(f"  ! {m} 호출 실패 ({type(e).__name__}): {str(e)[:120]}")
        if hit is None:
            raise RuntimeError(f"모델 호출 실패: {last_err}")
        _store(key, hit)
    return schema.model_validate_json(hit) if schema is not None else hit


def embed_texts(texts, model=None):
    """문장 목록을 벡터로 바꾼다. model='tfidf'면 API 없이 TF-IDF로 대신한다."""
    import numpy as np
    model = model or EMBED_MODEL
    if model == "tfidf":
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(stop_words="english").fit(texts)
        return np.asarray(vec.transform(texts).todense()), vec
    if _client is None:
        gemini_client()
    out = []
    for i in range(0, len(texts), 50):                 # 한 번에 최대 50개
        batch = texts[i:i + 50]
        key = hashlib.sha256((model + "\n" + "\x1e".join(batch)).encode("utf-8")).hexdigest()
        hit = _cached(key)
        if hit is None:
            _throttle()
            r = _client.models.embed_content(model=model, contents=batch)
            hit = [e.values for e in r.embeddings]
            _store(key, hit)
        out.extend(hit)
    return np.asarray(out, dtype="float32"), None


def save_json(name, obj):
    WORK.mkdir(parents=True, exist_ok=True)
    p = WORK / name
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return p


def download(paths, zip_name):
    """결과 파일을 zip으로 묶는다. Colab이면 내려받기 창을 띄운다."""
    import zipfile
    WORK.mkdir(parents=True, exist_ok=True)
    zp = WORK / zip_name
    with zipfile.ZipFile(zp, "w") as z:
        for p in paths:
            p = pathlib.Path(p)
            if p.exists():
                z.write(p, p.name)
    print("저장:", zp.resolve())
    if IN_COLAB:
        from google.colab import files
        files.download(str(zp))
    return zp


ensure_packages({'requests': 'requests', 'lxml': 'lxml', 'networkx': 'networkx', 'sklearn': 'scikit-learn', 'igraph': 'igraph', 'leidenalg': 'leidenalg'})

import warnings; warnings.filterwarnings('ignore')

import matplotlib; matplotlib.rcParams['figure.dpi'] = 100

# ── 미니 논문 3편 (실습용으로 작성한 가상 논문, 실제 논문 아님) ─────────────────────
# 실습 1의 추출·채점 대상이고, 실습 2에서는 Europe PMC에 접속할 수 없을 때 대신 쓰는 말뭉치다.
PAPERS = {
    "A": {
        "title": "Weakly solvating ether electrolytes for stable Li||NCM811 cells",
        "methods": (
            "Electrolytes were prepared by dissolving 1.0 M LiFSI in diethyl ether (DEE), dipropyl ether (DPE) or "
            "diethylene glycol dimethyl ether (DEGDME); the corresponding cells are denoted as the DEE-based cell, the "
            "DPE-based cell and the baseline cell. CR2032 coin cells were assembled with 50 µm Li foil, NCM811 cathodes "
            "(areal capacity 2.0 mAh cm−2), a Celgard 2325 separator and 40 µL of electrolyte. After two formation cycles "
            "at 0.1C, the cells were cycled at 1C between 2.8 and 4.4 V at 25 °C. Ionic conductivity was measured by "
            "electrochemical impedance spectroscopy at 25 °C."),
        "results": (
            "The DEE electrolyte showed an ionic conductivity of 4.2 mS cm−1, lower than that of the DEGDME electrolyte "
            "(7.9 mS cm−1). A previous study reported 10.6 mS cm−1 for a localized high-concentration electrolyte [21]. "
            "In the first formation cycle at 0.1C, the DEE-based cell delivered 205 mAh g−1. At 1C, the DEE-based cell "
            "showed an initial discharge capacity of 182 mAh g−1 and retained 94% of its capacity after 100 cycles, "
            "whereas the baseline cell retained only 66% (Fig. 2a). The average Coulombic efficiency of the DEE-based "
            "cell over 100 cycles was 99.6%."),
        "caption": (
            "Fig. 2 | Cycling performance of Li||NCM811 cells at 1C after formation. a, Discharge capacity (filled "
            "symbols, left axis) and Coulombic efficiency (open symbols, right axis). E1, E2 and Ref. denote the DEE, "
            "DPE and DEGDME electrolytes, respectively."),
    },
    "B": {
        "title": "A nitrate additive for dendrite-free Li||LFP cells",
        "methods": (
            "The baseline electrolyte was 1.0 M LiTFSI in a 1:1 (v/v) mixture of 1,3-dioxolane (DOL) and "
            "1,2-dimethoxyethane (DME). For the LiNO3-added electrolyte, 2 wt% LiNO3 was added to the baseline "
            "electrolyte. Li||LiFePO4 coin cells were cycled at 0.5C between 2.5 and 4.0 V."),
        "results": (
            "The LiNO3-added electrolyte exhibited an ionic conductivity of 1.05 S m−1, comparable to that of the "
            "additive-free electrolyte (1.12 S m−1). The LiNO3-added cell delivered an initial discharge capacity of "
            "152 mAh g−1 at 0.5C and retained 97% of its capacity after 200 cycles, while the additive-free cell "
            "retained 71%. Previous reports on LiNO3-containing carbonate electrolytes showed a retention of about "
            "80% [12]."),
        "caption": "Fig. 3 | Cycling stability of Li||LFP cells with and without the LiNO3 additive at 0.5C.",
    },
    "C": {
        "title": "Localized high-concentration electrolyte for 4.5 V Li metal cells",
        "methods": (
            "The localized high-concentration electrolyte (LHCE) consisted of LiFSI, DME and TTE in a 1:1.2:3 molar "
            "ratio, and the conventional electrolyte was 1.0 M LiPF6 in EC/DMC (1:1, v/v). Li||NCM622 cells were "
            "cycled between 3.0 and 4.5 V at 1C after one formation cycle at 0.1C."),
        "results": (
            "The ionic conductivity of the LHCE (1.6 mS cm−1) was lower than that of the conventional electrolyte "
            "(10.1 mS cm−1). The first-cycle discharge capacity of the LHCE cell at 0.1C was 198 mAh g−1, and its "
            "initial discharge capacity at 1C was 176 mAh g−1. After 300 cycles, the LHCE cell retained 89% of its "
            "initial capacity, whereas the conventional cell failed after 120 cycles. The LHCE cell maintained an "
            "average Coulombic efficiency of 99.7%, compared with 98.1% for the conventional cell."),
        "caption": "Fig. 4 | Long-term cycling of Li||NCM622 cells with the LHCE and the conventional electrolyte.",
    },
}


def paper_text(pid):
    p = PAPERS[pid]
    return f"Title: {p['title']}\n\nMethods: {p['methods']}\n\nResults: {p['results']}\n\nCaption: {p['caption']}"


def show_papers():
    for pid in PAPERS:
        print(f"[논문 {pid}] " + paper_text(pid).replace("\n\n", "\n") + "\n")


# ── 실습 2 · RAG와 GraphRAG로 논문에 질문하기 ─────────────────────────────────
# 논문: Europe PMC의 CC BY 오픈액세스 논문 5편을 실행할 때 내려받는다 (저장소에 넣지 않는다).
# 접속이 안 되면 mini_papers.py의 미니 논문 3편으로 대신 진행한다.
import json
import re
from typing import Optional

from pydantic import BaseModel, Field

PMCIDS = ["PMC9076822", "PMC6128910", "PMC7442789", "PMC9935899", "PMC9666536"]
MAX_CHARS = 10000          # 논문 한 편에서 쓸 글자 수 (호출 수·시간을 줄이려고 앞부분만)


# ── 2. 논문 불러오기 ─────────────────────────────────────────────────────
def _jats_text(el):
    return re.sub(r"\s+", " ", "".join(el.itertext())).strip()


def _fetch_pmc(pmcid, max_chars):
    import requests
    from lxml import etree
    r = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML", timeout=60)
    r.raise_for_status()
    root = etree.fromstring(r.content)
    title = _jats_text(root.find(".//article-title"))
    abstract = " ".join(_jats_text(p) for p in root.findall(".//abstract//p"))
    secs = {}
    for sec in root.findall(".//body/sec"):
        t = sec.find("title")
        name = (_jats_text(t) if t is not None else "").lower()
        key = next((k for k in ("results", "discussion", "methods") if k in name), None)
        if key:
            secs[key] = secs.get(key, "") + " " + " ".join(_jats_text(p) for p in sec.findall(".//p"))
    caps = " ".join(_jats_text(c) for c in root.findall(".//fig//caption"))
    rest = max(max_chars - len(abstract), 2000)
    budget = {"results": 0.45, "discussion": 0.15, "methods": 0.25, "captions": 0.15}
    parts = [("Abstract", abstract)]
    for key, share in budget.items():
        txt = caps if key == "captions" else secs.get(key, "")
        if txt.strip():
            parts.append((key.capitalize(), txt.strip()[: int(rest * share)]))
    return {"pmcid": pmcid, "title": title, "sections": parts}


def load_papers(pmcids=None, max_chars=None, use_europepmc=True):
    """논문을 불러온다. 실패하면 미니 논문으로 대신한다."""
    pmcids = pmcids or PMCIDS
    max_chars = max_chars or MAX_CHARS
    papers = []
    if use_europepmc:
        for i, pid in enumerate(pmcids, 1):
            try:
                p = _fetch_pmc(pid, max_chars)
                p["id"] = f"P{i}"
                papers.append(p)
            except Exception as e:
                print(f"  ! {pid} 내려받기 실패 ({type(e).__name__}) — 건너뜀")
    if not papers:
        print("Europe PMC에 접속하지 못해 실습용 미니 논문 3편으로 진행한다.")
        for i, pid in enumerate(PAPERS, 1):
            q = PAPERS[pid]
            papers.append({"id": f"M{pid}", "pmcid": f"mini-{pid}", "title": q["title"],
                           "sections": [("Methods", q["methods"]), ("Results", q["results"]), ("Captions", q["caption"])]})
    for p in papers:
        n = sum(len(t) for _, t in p["sections"])
        print(f"  {p['id']}  {p['pmcid']:11s}  {n:6,d}자  {p['title'][:70]}")
    return papers


# ── 3. 청킹 ──────────────────────────────────────────────────────────────
def split(papers, size=1000, overlap=200):
    """논문을 size 글자씩, overlap 글자씩 겹치게 자른다. 가능하면 문장 끝에서 자른다."""
    chunks = []
    for p in papers:
        text = "\n".join(f"[{name}] {t}" for name, t in p["sections"])
        i, n = 0, 0
        while i < len(text):
            end = min(i + size, len(text))
            if end < len(text):
                cut = text.rfind(". ", i + size // 2, end)
                end = cut + 1 if cut != -1 else end
            chunks.append({"id": f"{p['id']}-c{n}", "paper": p["id"], "title": p["title"], "text": text[i:end].strip()})
            n += 1
            if end >= len(text):
                break
            i = max(end - overlap, i + 1)
    print(f"청크 {len(chunks)}개 (크기 {size}자, 겹침 {overlap}자)")
    return chunks


# ── 4. 임베딩과 코사인 유사도 ──────────────────────────────────────────────
def embed(chunks, model=None):
    """청크마다 벡터를 만든다. model='tfidf'면 API 없이 TF-IDF로 한다."""
    import numpy as np
    model = model or EMBED_MODEL
    vecs, vectorizer = embed_texts([c["text"] for c in chunks], model)
    if model != "tfidf":
        vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
    print(f"임베딩: 청크 {len(chunks)}개 → {vecs.shape[1]}차원 벡터 ({model})")
    return {"chunks": chunks, "vecs": vecs, "model": model, "vectorizer": vectorizer}


def top_k(store, question, k=5):
    import numpy as np
    if store["model"] == "tfidf":
        q = np.asarray(store["vectorizer"].transform([question]).todense())[0]
        V = store["vecs"]
        sims = V @ q / ((np.linalg.norm(V, axis=1) + 1e-9) * (np.linalg.norm(q) + 1e-9))
    else:
        q = embed_texts([question], store["model"])[0][0]
        q = q / (np.linalg.norm(q) + 1e-9)
        sims = store["vecs"] @ q
    order = np.argsort(-sims)[:k]
    return [(float(sims[i]), store["chunks"][i]) for i in order]


def show_cosine(question, hits):
    import matplotlib.pyplot as plt
    import pandas as pd
    ids = [c["id"] for _, c in hits][::-1]
    vals = [s for s, _ in hits][::-1]
    plt.figure(figsize=(6, 0.45 * len(hits) + 1))
    plt.barh(ids, vals, color="#2F5597")
    plt.xlabel("cosine similarity")
    plt.title("Top chunks for the question")
    plt.tight_layout(); plt.show()
    _display(pd.DataFrame([{"청크": c["id"], "코사인": round(s, 3), "앞부분": c["text"][:110] + "…"} for s, c in hits]))


def ask(prompt, hits, question):
    """찾은 청크만 근거로 답한다. 사실마다 [청크 ID]를 붙이게 한다."""
    ctx = "\n\n".join(f"[{c['id']}] {c['text']}" for _, c in hits)
    answer = llm(f"{prompt.strip()}\n\n근거:\n{ctx}\n\n질문: {question}\n답 (한국어):")
    print(answer.strip())
    return answer.strip()


def _display(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string())


# ── 6. 개체와 관계 추출 (온톨로지 = ITEMS + RELATIONS) ─────────────────────────
ITEMS = """Electrolyte: 전해질 (이름 또는 염+용매 조합)
Salt: 리튬염 (LiFSI, LiPF6, LiNO3 등)
Solvent: 용매 (DME, DEE, EC 등)
Additive: 첨가제
Cathode: 양극 물질 (NCM811, LFP 등)
Anode: 음극 (Li metal 등)
Strategy: 성능을 높이는 설계 전략 (weakly solvating, fluorination, additive 등)
Phenomenon: 현상·메커니즘 (SEI 형성, 덴드라이트, 용매화 구조 등)
Property: 측정한 물성·성능과 값 (이온전도도, 쿨롱효율, 용량 유지율 등)"""

RELATIONS = """Electrolyte - CONTAINS - Salt
Electrolyte - CONTAINS - Solvent
Electrolyte - CONTAINS - Additive
Electrolyte - USES_STRATEGY - Strategy
Electrolyte - HAS_PROPERTY - Property
Electrolyte - TESTED_WITH - Cathode
Electrolyte - TESTED_WITH - Anode
Strategy - CAUSES - Phenomenon
Additive - CAUSES - Phenomenon
Phenomenon - AFFECTS - Property"""

ALIAS = {"diethylether": "dee", "dipropylether": "dpe", "12dimethoxyethane": "dme", "13dioxolane": "dol",
         "lithiumbisfluorosulfonylimide": "lifsi", "lithiumbistrifluoromethanesulfonylimide": "litfsi",
         "lithiumnitrate": "lino3", "lithiummetal": "limetal", "lithiummetalanode": "limetal", "lianode": "limetal",
         "solidelectrolyteinterphase": "sei", "solidelectrolyteinterphasesei": "sei"}


def _types():
    return [l.split(":")[0].strip() for l in ITEMS.strip().splitlines() if ":" in l]


def _rules():
    out = set()
    for l in RELATIONS.strip().splitlines():
        parts = [x.strip() for x in l.split(" - ")]
        if len(parts) == 3:
            out.add(tuple(parts))
    return out


def norm_name(name):
    k = re.sub(r"[^a-z0-9]", "", (name or "").lower())
    return ALIAS.get(k, k)


class Entity(BaseModel):
    name: str
    type: str = Field(description="ITEMS에 있는 종류 중 하나")
    description: str = Field(description="이 개체에 대해 글에 나온 내용 한 문장")


class Relation(BaseModel):
    source: str
    relation: str = Field(description="RELATIONS에 있는 관계 이름 중 하나")
    target: str
    evidence: str = Field(description="관계가 나온 원문 문장")


class GraphOut(BaseModel):
    entities: list[Entity]
    relations: list[Relation]


GRAPH = {"entities": {}, "relations": []}


def _add_entity(name, typ, desc, paper):
    k = norm_name(name)
    if not k:
        return None
    e = GRAPH["entities"].setdefault(k, {"key": k, "name": name, "type": typ, "descriptions": [], "papers": set()})
    if desc and len(e["descriptions"]) < 6:
        e["descriptions"].append(desc)
    e["papers"].add(paper)
    return k


def extract_graph(chunks):
    """청크마다 LLM이 온톨로지에 맞는 개체·관계를 뽑는다. 규칙에 없는 관계는 코드로 버린다."""
    GRAPH["entities"].clear(); GRAPH["relations"].clear()
    types, rules = _types(), _rules()
    dropped = 0
    for c in chunks:
        q = ("Extract entities and relations from the text using ONLY this schema.\n"
             f"Entity types:\n{ITEMS}\n\nAllowed relations (source type - RELATION - target type):\n{RELATIONS}\n\n"
             "Rules: use specific names as written; keep values and units in Property names "
             "(e.g. 'Coulombic efficiency 99.6%'); do not infer facts not stated in the text.\n\n"
             f"TEXT [{c['id']}]:\n{c['text']}")
        out = llm(q, schema=GraphOut)
        tmap = {}
        for e in out.entities:
            if e.type in types:
                k = _add_entity(e.name, e.type, e.description, c["paper"])
                tmap[norm_name(e.name)] = e.type
        for r in out.relations:
            s, t = norm_name(r.source), norm_name(r.target)
            if (tmap.get(s), r.relation, tmap.get(t)) not in rules:
                dropped += 1
                continue
            GRAPH["relations"].append({"source": s, "relation": r.relation, "target": t, "evidence": r.evidence,
                                       "chunk": c["id"], "paper": c["paper"]})
    print(f"추출: 개체 {len(GRAPH['entities'])}개, 관계 {len(GRAPH['relations'])}개 "
          f"(온톨로지 규칙에 맞지 않아 버린 관계 {dropped}개)")
    return GRAPH["entities"], GRAPH["relations"]


DEFAULT_RECORDS = [   # 실습 1의 정답 (records.csv가 없을 때 쓴다)
    ("DEE electrolyte", "ionic_conductivity", "4.2 mS/cm"), ("DEGDME electrolyte", "ionic_conductivity", "7.9 mS/cm"),
    ("DEE electrolyte", "capacity_retention", "94 %"), ("DEGDME electrolyte", "capacity_retention", "66 %"),
    ("LiNO3-added electrolyte", "capacity_retention", "97 %"), ("LHCE", "average_coulombic_efficiency", "99.7 %"),
]


def add_records(relations, path="records.csv"):
    """실습 1에서 만든 레코드를 그래프에 더한다 (Electrolyte - HAS_PROPERTY - Property)."""
    import pathlib
    rows = []
    for p in (pathlib.Path(path), WORK / path, pathlib.Path("/content") / path):
        if p.exists():
            import pandas as pd
            df = pd.read_csv(p)
            for _, r in df.iterrows():
                if r.get("property") in ("other", None) or str(r.get("property")) == "nan":
                    continue
                v = r.get("value_std", r.get("value")); u = r.get("unit_std", r.get("unit"))
                rows.append((str(r["material"]), str(r["property"]), f"{v} {u}"))
            print(f"실습 1 결과 {p} 에서 레코드 {len(rows)}개를 읽었다.")
            break
    if not rows:
        rows = DEFAULT_RECORDS
        print(f"records.csv가 없어 실습 1 정답 레코드 {len(rows)}개를 대신 넣는다.")
    for mat, prop, val in rows:
        s = _add_entity(mat, "Electrolyte", "실습 1 레코드", "Practice1")
        t = _add_entity(f"{prop.replace('_', ' ')} {val}", "Property", "실습 1 레코드", "Practice1")
        relations.append({"source": s, "relation": "HAS_PROPERTY", "target": t, "evidence": "실습 1 records.csv",
                          "chunk": "Practice1", "paper": "Practice1"})
    return relations


def show_graph_summary(entities, relations):
    import collections
    import matplotlib.pyplot as plt
    import pandas as pd
    cnt = collections.Counter(e["type"] for e in entities.values())
    deg = collections.Counter()
    for r in relations:
        deg[r["source"]] += 1; deg[r["target"]] += 1
    shared = [e for e in entities.values() if len(e["papers"] - {"Practice1"}) >= 2]
    plt.figure(figsize=(6, 3))
    names = sorted(cnt, key=cnt.get)
    plt.barh(names, [cnt[n] for n in names], color="#2F5597")
    plt.title("Entities by type"); plt.tight_layout(); plt.show()
    print(f"개체 {len(entities)}개 · 관계 {len(relations)}개 · 두 편 이상에 나온 개체 {len(shared)}개")
    top = [{"개체": entities[k]["name"], "종류": entities[k]["type"], "관계 수": d,
            "논문": ", ".join(sorted(entities[k]["papers"]))} for k, d in deg.most_common(10) if k in entities]
    _display(pd.DataFrame(top))


# ── 7. Leiden 커뮤니티 ───────────────────────────────────────────────────
def _nx_graph(entities, relations):
    import networkx as nx
    G = nx.Graph()
    for k, e in entities.items():
        G.add_node(k, type=e["type"])
    for r in relations:
        if r["source"] in G and r["target"] in G and r["source"] != r["target"]:
            G.add_edge(r["source"], r["target"])
    G.remove_nodes_from([n for n in list(G) if G.degree(n) == 0])
    return G


def find_communities(entities, relations, max_communities=8, min_size=3):
    """연결만 보고 촘촘한 묶음(커뮤니티)을 찾는다. Leiden이 없으면 Louvain으로 대신한다."""
    import networkx as nx
    G = _nx_graph(entities, relations)
    try:
        import igraph as ig
        import leidenalg
        nodes = list(G.nodes)
        idx = {n: i for i, n in enumerate(nodes)}
        g = ig.Graph(n=len(nodes), edges=[(idx[a], idx[b]) for a, b in G.edges])
        part = leidenalg.find_partition(g, leidenalg.ModularityVertexPartition, seed=1)
        groups = [[nodes[i] for i in c] for c in part]
        algo = "Leiden"
    except Exception:
        groups = [list(c) for c in nx.community.louvain_communities(G, seed=1)]
        algo = "Louvain (leidenalg 없음)"
    groups = sorted([g for g in groups if len(g) >= min_size], key=len, reverse=True)[:max_communities]
    comms = [{"id": f"C{i}", "nodes": g} for i, g in enumerate(groups)]
    print(f"{algo}: 커뮤니티 {len(comms)}개 (노드 {min_size}개 이상, 큰 순서로 최대 {max_communities}개)")
    for c in comms:
        print(f"  {c['id']}: {len(c['nodes'])}개 — " + ", ".join(entities[n]["name"] for n in c["nodes"][:6]))
    _plot_communities(G, comms, entities)
    return comms


def _plot_communities(G, comms, entities):
    import matplotlib.pyplot as plt
    import networkx as nx
    pal = ["#2F5597", "#ED7D31", "#548235", "#7030A0", "#C00000", "#00B0F0", "#FFC000", "#7F7F7F"]
    color = {}
    for i, c in enumerate(comms):
        for n in c["nodes"]:
            color[n] = pal[i % len(pal)]
    pos = nx.spring_layout(G, seed=3, k=0.35)
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_edges(G, pos, edge_color="#D0D0D0", width=0.6)
    nx.draw_networkx_nodes(G, pos, node_color=[color.get(n, "#E0E0E0") for n in G], node_size=60)
    for i, c in enumerate(comms):
        hub = max(c["nodes"], key=G.degree)
        x, y = pos[hub]
        plt.text(x, y, f"C{i}: {entities[hub]['name'][:22]}", fontsize=8, color=pal[i % len(pal)], fontweight="bold")
    plt.axis("off"); plt.title("Communities (grey = not in a reported community)"); plt.show()


# ── 8. 커뮤니티 요약 ─────────────────────────────────────────────────────
class Report(BaseModel):
    title: str
    summary: str
    findings: list[str]


def summarize(communities, entities=None, relations=None):
    """커뮤니티마다 LLM이 제목·요약·핵심 발견을 쓴다. GraphRAG는 원문이 아니라 이 보고서에서 답한다."""
    import pandas as pd
    entities = entities or GRAPH["entities"]
    relations = relations or GRAPH["relations"]
    reports = []
    for c in communities:
        nodes = set(c["nodes"])
        ents = "\n".join(f"- {entities[n]['name']} ({entities[n]['type']}): {'; '.join(entities[n]['descriptions'][:2])}"
                         for n in c["nodes"])
        rels = [r for r in relations if r["source"] in nodes and r["target"] in nodes][:40]
        rtxt = "\n".join(f"- {entities[r['source']]['name']} {r['relation']} {entities[r['target']]['name']} "
                         f"[{r['paper']}] \"{r['evidence'][:150]}\"" for r in rels)
        q = ("Write a report about this group of related entities from battery electrolyte papers. "
             "Use only the information below. Write in Korean.\n\n"
             f"Entities:\n{ents}\n\nRelations (with source paper):\n{rtxt}")
        rep = llm(q, schema=Report)
        papers = sorted({r["paper"] for r in rels} | {p for n in nodes for p in entities[n]["papers"]})
        reports.append({"id": c["id"], "title": rep.title, "summary": rep.summary, "findings": rep.findings,
                        "papers": papers, "size": len(nodes)})
    _display(pd.DataFrame([{"커뮤니티": r["id"], "개체 수": r["size"], "제목": r["title"],
                            "요약": r["summary"][:120] + "…", "논문": ", ".join(r["papers"])} for r in reports]))
    return reports


# ── 9. Map-Reduce 답 ─────────────────────────────────────────────────────
class Partial(BaseModel):
    answer: str
    score: int = Field(description="이 답이 질문에 얼마나 도움이 되는지 0–100")


def map_step(question, reports):
    """커뮤니티 보고서마다 따로 답하고 도움 점수를 매긴다."""
    import matplotlib.pyplot as plt
    out = []
    for r in reports:
        q = ("Answer the question using only this community report. If the report is not relevant, "
             "give score 0. Write in Korean.\n\n"
             f"Report {r['id']} — {r['title']}\n{r['summary']}\nFindings: {r['findings']}\n\nQuestion: {question}")
        p = llm(q, schema=Partial)
        out.append({"id": r["id"], "title": r["title"], "answer": p.answer, "score": p.score, "papers": r["papers"]})
    out.sort(key=lambda x: -x["score"])
    plt.figure(figsize=(6, 0.4 * len(out) + 1))      # 축에는 ID만 (한글 글꼴 없이도 보이게), 제목은 아래 표로
    plt.barh([x["id"] for x in out][::-1], [x["score"] for x in out][::-1], color="#2F5597")
    plt.xlabel("helpfulness score (map step)"); plt.xlim(0, 100); plt.tight_layout(); plt.show()
    import pandas as pd
    _display(pd.DataFrame([{"커뮤니티": x["id"], "점수": x["score"], "제목": x["title"]} for x in out]))
    return out


def reduce_step(question, partials, top=5, min_score=20):
    """점수가 높은 부분 답만 모아 하나의 답으로 합친다. [C번호]로 출처를 남긴다."""
    use = [p for p in partials if p["score"] >= min_score][:top]
    if not use:
        print("도움이 되는 커뮤니티가 없다 — 그래프에 관련 정보가 없을 수 있다.")
        return ""
    ctx = "\n\n".join(f"[{p['id']}] (score {p['score']}, papers {', '.join(p['papers'])}) {p['answer']}" for p in use)
    ans = llm("Combine these partial answers into one answer. Cite community ids like [C0] after each claim. "
              f"Write in Korean.\n\nPartial answers:\n{ctx}\n\nQuestion: {question}").strip()
    print(ans)
    return ans


# ── 10–11. 비교와 판정 ────────────────────────────────────────────────────
RAG_PROMPT = """아래 근거만으로 답한다. 사실마다 [청크 ID]를 붙인다.
근거가 부족하면 "근거 없음"이라고 쓴다."""


def compare(questions, reports, store, k=5):
    """같은 질문을 기존 RAG와 GraphRAG에 던져 답을 나란히 놓는다."""
    import pandas as pd
    rows = []
    for q in [q for q in questions if q and q.strip()]:
        print(f"\n■ 질문: {q}\n— 기존 RAG —")
        hits = top_k(store, q, k)
        v = ask(RAG_PROMPT, hits, q)
        print("— GraphRAG —")
        g = reduce_step(q, map_step(q, reports))
        rows.append({"question": q, "vector_answer": v, "graph_answer": g,
                     "vector_sources": ", ".join(c["id"] for _, c in hits)})
    if not rows:
        print("MY_QUESTIONS에 질문을 적어야 한다.")
    return pd.DataFrame(rows)


def save_comparison(comparison, better, reasons):
    """판정(graph / vector / tie / neither)과 이유를 rag_comparison.md로 저장한다. 모델 호출 없음."""
    import pandas as pd
    df = comparison.copy()
    df["better_supported"] = (list(better) + ["unreviewed"] * len(df))[:len(df)]
    df["reason"] = (list(reasons) + [""] * len(df))[:len(df)]
    lines = ["# RAG vs GraphRAG 비교", ""]
    for i, r in df.iterrows():
        lines += [f"## 질문 {i + 1}. {r['question']}", "", f"**판정:** {r['better_supported']}",
                  f"**이유:** {r['reason']}", "", "### 기존 RAG", r["vector_answer"], "",
                  f"출처 청크: {r['vector_sources']}", "", "### GraphRAG", r["graph_answer"], ""]
    p = WORK / "rag_comparison.md"
    WORK.mkdir(parents=True, exist_ok=True)
    p.write_text("\n".join(lines), encoding="utf-8")
    print("저장:", p.resolve())
    _display(df[["question", "better_supported", "reason"]])
    return p


def export_results():
    files = [WORK / "rag_comparison.md"]
    save_json("graph.json", {"entities": [{**e, "papers": sorted(e["papers"])} for e in GRAPH["entities"].values()],
                             "relations": GRAPH["relations"]})
    save_json("ontology.json", {"items": ITEMS, "relations": RELATIONS})
    files += [WORK / "graph.json", WORK / "ontology.json"]
    return download(files, "p2_results.zip")


print('준비 완료 ·', 'Colab' if IN_COLAB else '로컬', '· 작업 폴더', WORK.resolve())

## Gemini API 키 확인

키는 **각자 발급**해서 쓴다. 발급: <https://aistudio.google.com/apikey> → **Create API key** (개인 구글 계정, 무료).

| 환경 | 키를 넣는 곳 |
|---|---|
| Colab | 왼쪽 **열쇠 아이콘(보안 비밀)** → 이름 `GEMINI_API_KEY`, 값에 키 → **노트북 액세스** 켜기 |
| 로컬 (Jupyter, VS Code) | 노트북과 같은 폴더나 상위 폴더의 `.env` 파일에 `GEMINI_API_KEY=키` 한 줄 |
| 둘 다 없으면 | 아래 셀을 실행하면 입력창이 뜬다 (화면에 보이지 않음) |

키는 비밀번호와 같다. 셀에 직접 적거나 공유하지 않는다. `키 확인: 찾음`이 나오면 된다.

In [ ]:
client = gemini_client()           # '키 확인: 찾음'이 나와야 한다
MODEL = "gemini-3.5-flash-lite"    # 막히면 "gemini-3.1-flash-lite"
EMBED_MODEL = "gemini-embedding-001"   # 임베딩이 막히면 "tfidf" (API 없이 진행)

## 2. 논문 불러오기

리튬금속전지 전해질 논문 5편 (Nature Communications, CC BY). `MAX_CHARS`는 논문마다 쓸 글자 수다 — 초록, 결과, 논의, 방법, 그림 설명을 나눠 담는다.

In [ ]:
USE_EUROPEPMC = True   # False = 실습용 미니 논문 3편
MAX_CHARS = 10000      # 논문 한 편에서 쓸 글자 수
papers = load_papers(PMCIDS, MAX_CHARS, USE_EUROPEPMC)

## 3. 청킹과 임베딩 (기존 RAG의 색인)

In [ ]:
chunks = split(papers, size=1000, overlap=200)
store = embed(chunks, model=EMBED_MODEL)

## 4. 질문 임베딩과 코사인 유사도

질문을 같은 방식으로 벡터로 바꾸고, 가장 가까운 청크 k개를 찾는다. 질문을 바꿔 보자.

In [ ]:
QUESTION = "LiHMDS 첨가제는 전해질에서 어떤 역할을 하는가?"
hits = top_k(store, QUESTION, k=5)
show_cosine(QUESTION, hits)

## 5. 답 생성 — 찾은 근거로만 답하고, 출처를 붙인다

답 뒤의 `[P5-c1]` 같은 표시가 근거 청크다. 위 표에서 그 청크를 찾아 답이 맞는지 직접 확인해 보자.

**해 볼 것**: 4번의 `QUESTION`을 `"What cycling conditions (C-rate, voltage range) were used for the fluorinated ether electrolyte cells?"`로 바꿔 4–5번을 다시 실행한다.
`근거 없음`이 나오면, 강의의 **RAG 실패 지점** 중 어느 것인지 생각해 보자 (답이 상위 k개 밖? 논문 앞부분만 잘라서 문서에 없음?).

In [ ]:
PROMPT = """아래 근거만으로 답한다. 사실마다 [청크 ID]를 붙인다.
근거가 부족하면 "근거 없음"이라고 쓴다."""
answer = ask(PROMPT, hits, QUESTION)

## 온톨로지 정하기 — 무엇을 노드로, 어떤 관계를 허용할지

`ITEMS`는 노드 종류, `RELATIONS`는 허용된 관계(`출발 종류 - 관계 - 도착 종류`)다.
규칙에 없는 관계는 LLM이 뽑아도 **코드가 버린다**. 내 연구 주제에 맞게 고쳐 봐도 된다 (고치면 6번부터 다시 실행).

In [ ]:
ITEMS = """
Electrolyte: 전해질 (이름 또는 염+용매 조합)
Salt: 리튬염 (LiFSI, LiPF6, LiNO3 등)
Solvent: 용매 (DME, DEE, EC 등)
Additive: 첨가제
Cathode: 양극 물질 (NCM811, LFP 등)
Anode: 음극 (Li metal 등)
Strategy: 성능을 높이는 설계 전략 (weakly solvating, fluorination, additive 등)
Phenomenon: 현상·메커니즘 (SEI 형성, 덴드라이트, 용매화 구조 등)
Property: 측정한 물성·성능과 값 (이온전도도, 쿨롱효율, 용량 유지율 등)
"""
RELATIONS = """
Electrolyte - CONTAINS - Salt
Electrolyte - CONTAINS - Solvent
Electrolyte - CONTAINS - Additive
Electrolyte - USES_STRATEGY - Strategy
Electrolyte - HAS_PROPERTY - Property
Electrolyte - TESTED_WITH - Cathode
Electrolyte - TESTED_WITH - Anode
Strategy - CAUSES - Phenomenon
Additive - CAUSES - Phenomenon
Phenomenon - AFFECTS - Property
"""

## 6. 개체와 관계 추출

청크를 크게(4,000자) 잘라 청크마다 LLM을 한 번 부른다. 실습 1의 `records.csv`가 있으면 그래프에 더한다
(Colab은 왼쪽 **폴더 아이콘**에 `records.csv`를 올린다. 없으면 실습 1 정답 레코드를 대신 넣는다).

**두 편 이상에 나온 개체** 수를 보자 — 이 수가 적으면 논문 사이가 연결되지 않는다 (개체 병합 문제).

**찾아볼 것**: 같은 물질인데 다른 노드로 남은 것이 있나? (예: `NCM 811`과 `LiNi0.8Co0.1Mn0.1O2`, `DPE electrolyte`와 `DPE-based electrolyte`)
실습 1 레코드는 논문 그래프와 이어졌나, 따로 떨어져 있나? 이어지려면 무엇이 같아야 할까?

In [ ]:
graph_chunks = split(papers, size=4000, overlap=200)
entities, relations = extract_graph(graph_chunks)
add_records(relations, 'records.csv')   # 실습 1 결과
show_graph_summary(entities, relations)

## 7. Leiden 커뮤니티 — 연결만 보고 주제 묶음을 찾는다

In [ ]:
communities = find_communities(entities, relations)

## 8. 커뮤니티 요약 — GraphRAG가 검색하는 것은 원문이 아니라 이 보고서다

In [ ]:
reports = summarize(communities)

## 9. Map-Reduce로 답하기

**Map**: 보고서마다 따로 답하고 도움 점수(0–100)를 매긴다. **Reduce**: 점수가 높은 답만 합친다.

In [ ]:
QUESTION_GLOBAL = "이 논문들은 리튬금속전지 전해질을 개선하기 위해 어떤 전략을 쓰는가?"
partials = map_step(QUESTION_GLOBAL, reports)
graph_answer = reduce_step(QUESTION_GLOBAL, partials)

## 10. 내 질문 세 개

세 종류로 하나씩 적는다: **① 논문 전체에 대한 질문**, **② 단일 사실**, **③ 두 논문 비교**.
같은 질문이 기존 RAG와 GraphRAG에 모두 들어간다.

**"근거 없음"이 나오면**
- 지어내지 않도록 일부러 그렇게 지시했다: 근거에서 답을 못 찾으면 RAG는 **"근거 없음"**, GraphRAG는 **"도움이 되는 커뮤니티가 없다"**고 답한다 (Map 점수 막대가 모두 0)
- 못 찾는 이유
  - 논문마다 **앞부분만** 쓴다 (`MAX_CHARS = 10000`, 전체의 약 20–30%) → 답이 뒤쪽에 있으면 처음부터 없다
  - 기존 RAG는 질문과 비슷한 청크 **5개만** 본다 → 답이 있는 청크가 6위 밖이면 못 본다
  - GraphRAG는 원문이 아니라 **커뮤니티 요약**에서 답한다 → 구체적인 숫자는 요약에서 빠지기 쉽다
- 논문 전체를 묻는 질문(①)은 GraphRAG가, 특정 숫자(②)는 둘 다 약하다. "근거 없음"도 판정의 근거가 된다

In [ ]:
MY_QUESTIONS = [
    "",   # 1. 논문 전체에 대한 질문
    "",   # 2. 단일 사실
    "",   # 3. 두 논문 비교
]
comparison = compare(MY_QUESTIONS, reports, store)

## 11. 판정하고 저장 — 모델 호출 없음

질문마다 `graph`, `vector`, `tie`, `neither` 중 하나와 **근거에 기반한 이유 한 줄**을 적는다.

In [ ]:
BETTER  = ['unreviewed', 'unreviewed', 'unreviewed']   # graph / vector / tie / neither
REASONS = ['', '', '']
save_comparison(comparison, BETTER, REASONS)

## 12. 결과 내려받기 — `rag_comparison.md`, `graph.json`, `ontology.json`

In [ ]:
export_results()

## 정리

- 기존 RAG: 청크 하나 = 벡터 하나. **몇 개 문단에 있는 구체적 사실**에 강하다
- GraphRAG: 개체·관계 → 커뮤니티 → 요약 → map-reduce. **문서 전체를 아우르는 질문**에 강하다
- 둘 다 답은 **근거까지 추적**할 수 있어야 한다 (청크 ID, 커뮤니티 ID → 논문)
- 그래프의 품질은 **추출과 개체 병합의 품질**이다 — 실습 1의 스키마·검증이 그대로 중요하다